# DBScan Clustering with PyCaret

## Assignment D
**Goal:** Implement DBScan clustering using the PyCaret library.

**Dataset:** [Glass Identification](https://paperswithcode.com/dataset/glass-identification) (loaded via `pycaret.datasets` or `sklearn` if not available directly, but we will use `pycaret`'s built-in data for simplicity or load external if needed. Here we use `glass` from PyCaret if available, otherwise we load from URL).

In [ ]:
!pip install -q pycaret

In [ ]:
import pandas as pd
from pycaret.clustering import *

# Load Glass dataset from PyCaret repository or URL
# PyCaret has a 'glass' dataset in its repository
from pycaret.datasets import get_data
data = get_data('glass')

# Check data
print(data.head())

## 1. Setup PyCaret Environment
We initialize the clustering environment. We ignore the 'Type' column as we want to cluster unsupervised.

In [ ]:
s = setup(data, ignore_features=['Type'], session_id=123, verbose=False)
print("Setup Complete")

## 2. Create DBScan Model
We create the DBScan model. Note that DBScan does not take `num_clusters` as a parameter, but `eps` and `min_samples`.

In [ ]:
dbscan = create_model('dbscan', eps=0.5, min_samples=5)
print(dbscan)

## 3. Assign Labels
We assign the cluster labels to the original dataset.

In [ ]:
results = assign_model(dbscan)
results.head()

## 4. Visualization
PyCaret provides easy plotting functions.

In [ ]:
# 2D Plot (PCA)
plot_model(dbscan, plot='cluster')

# Distribution Plot
plot_model(dbscan, plot='distribution')

## 5. Evaluation
We can check the Silhouette Score which is automatically calculated by PyCaret during model creation, or calculate it manually.

In [ ]:
from sklearn.metrics import silhouette_score
X = get_config('X')
labels = results['Cluster']

# Filter out noise points (-1) for silhouette score if desired, or keep them
# DBScan labels noise as 'Cluster -1' usually, or PyCaret might map it.
# Let's check unique labels
print(f"Unique clusters: {labels.unique()}")

# Calculate score (excluding noise if needed, but standard is to include or handle separately)
# Here we calculate for all points
# Note: PyCaret labels might be strings like 'Cluster 0', 'Cluster 1'
# We need to encode them for sklearn metrics if they are strings
if labels.dtype == 'object':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    labels_encoded = le.fit_transform(labels)
else:
    labels_encoded = labels

score = silhouette_score(X, labels_encoded)
print(f"Silhouette Score: {score:.4f}")